In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow as pa 

In [2]:
gpx= pd.read_csv(r'C:\Users\nath2\OneDrive\Bureau\csvgarmin\activities.csv')

In [3]:
from google.cloud import bigquery

client = bigquery.Client()

dataset_id = f'{client.project}.running_data'

dataset= bigquery.Dataset(dataset_id)
dataset.location = 'US'

dataset= client.create_dataset(dataset, exists_ok=True)

print(f'Dataset créé (ou déjà existant) : {dataset.dataset_id}  ')
print(f'Projet : {dataset.project} ')
print(f'Localisation : {dataset.location} ')

Dataset créé (ou déjà existant) : running_data  
Projet : gpxstrava-510018 
Localisation : US 


In [4]:
df = pd.read_csv(r'C:\Users\nath2\OneDrive\Bureau\csvgarmin\activities.csv')
print(df.shape)
print(df.dtypes)
print(df.head(3))
print(df["Type d'activité"].value_counts())

(60, 40)
Type d'activité                                   str
Date                                              str
Favori                                           bool
Titre                                             str
Distance                                      float64
Calories                                          str
Durée                                             str
Fréquence cardiaque moyenne                     int64
Fréquence cardiaque maximale                    int64
TE aérobie                                    float64
Cadence de course moyenne                       int64
Cadence de course maximale                      int64
Allure moyenne                                    str
Meilleure allure                                  str
Ascension totale                                  str
Descente totale                                   str
Longueur moyenne des foulées                  float64
Rapport vertical moyen                        float64
Oscillation vertica

In [5]:
print(df.columns.tolist())

["Type d'activité", 'Date', 'Favori', 'Titre', 'Distance', 'Calories', 'Durée', 'Fréquence cardiaque moyenne', 'Fréquence cardiaque maximale', 'TE aérobie', 'Cadence de course moyenne', 'Cadence de course maximale', 'Allure moyenne', 'Meilleure allure', 'Ascension totale', 'Descente totale', 'Longueur moyenne des foulées', 'Rapport vertical moyen', 'Oscillation verticale moyenne', 'Temps de contact moyen avec le sol', 'Équilibre du temps de contact au sol moyen', 'GAP moyenne', 'Normalized Power® (NP®)', 'Training Stress Score® (TSS®)', 'Puissance moyenne', 'Puissance max.', 'Pas', 'Consommation du Body Battery', 'Température minimale', 'Décompression', 'Temps du meilleur circuit', 'Nombre de tours', 'Température maximale', 'Fréquence respiratoire moyenne', 'Fréquence respiratoire minimale', 'Fréquence respiratoire maximale', 'Temps de déplacement', 'Temps écoulé', 'Altitude minimale', 'Altitude maximale']


In [6]:
colonnes_utiles = {
   "Type d'activité": "type_activite",
    "Date": "date",
    "Titre": "titre",
    "Distance": "distance_km",
    "Durée": "duree",
    "Fréquence cardiaque moyenne": "fc_moyenne",
    "Fréquence cardiaque maximale": "fc_max",
    "Allure moyenne": "allure_moyenne",
    "Ascension totale": "denivele_positif",
    "Calories": "calories",
}

df=df[list(colonnes_utiles.keys())].rename(columns=colonnes_utiles)
print(df.head())

   type_activite                 date                  titre  distance_km  \
0  Course à pied  2026-09-23 19:32:52  Avignon Course à pied         6.42   
1  Course à pied  2026-09-16 19:33:46  Avignon Course à pied         7.66   
2          Trail  2026-09-13 11:20:35        Mormoiron Trail         6.02   
3  Course à pied  2026-09-09 19:38:23  Avignon Course à pied         7.42   
4          Trail  2026-09-07 18:42:24        Mormoiron Trail         6.01   

      duree  fc_moyenne  fc_max allure_moyenne denivele_positif calories  
0  00:39:50         149     202           6:12               24      428  
1  00:47:41         159     179           6:13               22      532  
2  00:40:23         156     203           6:42              169      428  
3  00:51:06         161     208           6:53              135      557  
4  00:43:28         152     170           7:14              173      422  


In [7]:
#date : texte -> datetime
df['date'] = pd.to_datetime(df["date"])


#distance et dénivelé : 
df['distance_km'] = pd.to_numeric(df["distance_km"], errors="coerce")
df['denivele_positif'] = pd.to_numeric(df["denivele_positif"], errors="coerce")


# Durée 'hh:mm:ss' -> secondes
def duree_en_secondes(duree_str):
    h,m,s = duree_str.split(":")
    return int(h)*3600 + int(m)*60 + float(s)

df['duree_secondes'] = df['duree'].apply(duree_en_secondes)

#Allure 
def allure_en_secondes(allure_str):
    if allure_str in ("--", None) or pd.isna(allure_str):
        return None
    m, s = allure_str.split(":")
    return int(m) + int(s) / 60

df["allure_moyenne_min_km"] = df["allure_moyenne"].apply(allure_en_secondes)

print(df.dtypes)
print(df.isna().sum())


type_activite                       str
date                     datetime64[us]
titre                               str
distance_km                     float64
duree                               str
fc_moyenne                        int64
fc_max                            int64
allure_moyenne                      str
denivele_positif                float64
calories                            str
duree_secondes                  float64
allure_moyenne_min_km           float64
dtype: object
type_activite            0
date                     0
titre                    0
distance_km              0
duree                    0
fc_moyenne               0
fc_max                   0
allure_moyenne           0
denivele_positif         2
calories                 0
duree_secondes           0
allure_moyenne_min_km    0
dtype: int64


In [8]:
df["semaine"] = df["date"].dt.to_period("W").astype(str)
df["mois"] = df["date"].dt.to_period("M").astype(str)
df["duree_minutes"] = df["duree_secondes"] / 60

In [9]:
from google.cloud import bigquery
import io

client = bigquery.Client()
table_id = f"{client.project}.running_data.activities"

# On encode directement en bytes UTF-8 plutôt qu'en texte
csv_buffer = io.BytesIO()
csv_buffer.write(df.to_csv(index=False).encode("utf-8"))
csv_buffer.seek(0)

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition="WRITE_TRUNCATE",
)

job = client.load_table_from_file(csv_buffer, table_id, job_config=job_config)

try:
    job.result()
except Exception as e:
    print(repr(e))
else:
    table = client.get_table(table_id)
    print(f"{table.num_rows} lignes chargées dans {table_id}")

60 lignes chargées dans gpxstrava-510018.running_data.activities
